# 02 · EDA — both datasets, hosted on HF

Reads both CSVs from the HF dataset repo created in notebook **01**
(`<HF_USERNAME>/fraud-detection-datasets`), then produces the diagnostic plots
and statistics needed to answer the study's research questions:

- **What features have the strongest predictive power for fraud?**
- **How severe is the class imbalance?**
- **Are there device / location / time patterns worth engineering?**

In [ ]:
import sys, subprocess
for p in ('huggingface_hub','pandas','seaborn','matplotlib'):
    try: __import__(p)
    except ImportError: subprocess.check_call([sys.executable,'-m','pip','install','-q',p])
print('deps OK')

In [ ]:
import os
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from huggingface_hub import hf_hub_download, whoami

HF_TOKEN = os.environ['HF_TOKEN']
HF_USERNAME = os.environ.get('HF_USERNAME') or whoami(token=HF_TOKEN)['name']
DATASET_REPO = f'{HF_USERNAME}/fraud-detection-datasets'

paysim_csv = hf_hub_download(repo_id=DATASET_REPO, filename='paysim.csv', repo_type='dataset', token=HF_TOKEN)
custom_csv = hf_hub_download(repo_id=DATASET_REPO, filename='custom.csv', repo_type='dataset', token=HF_TOKEN)

paysim = pd.read_csv(paysim_csv)
custom = pd.read_csv(custom_csv)
print(f'PaySim: {paysim.shape}, fraud rate {paysim.isFraud.mean():.4%}')
print(f'Custom: {custom.shape}, fraud rate {custom.Fraudulent.mean():.4%}')

## PaySim — fraud by transaction type

In [ ]:
by_type = paysim.groupby('type')['isFraud'].agg(['count', 'mean']).rename(columns={'mean': 'fraud_rate'})
by_type.sort_values('fraud_rate', ascending=False)

In [ ]:
plt.figure(figsize=(10, 5))
sns.boxplot(data=paysim, x='isFraud', y='amount', hue='type')
plt.yscale('log')
plt.title('Transaction amount — legit vs fraud, by type')
plt.show()

## PaySim — engineered features

Three signals that domain knowledge suggests will be informative.

In [ ]:
paysim['amount_eq_orig'] = (abs(paysim.amount - paysim.oldbalanceOrg) < 1.0).astype(int)
paysim['dest_was_empty']  = (paysim.oldbalanceDest == 0).astype(int)
paysim['delta_orig']      = paysim.oldbalanceOrg - paysim.newbalanceOrig

for col in ['amount_eq_orig', 'dest_was_empty']:
    print(f'{col} vs fraud:')
    print(pd.crosstab(paysim[col], paysim.isFraud, normalize='index'))
    print()

## PaySim — correlation matrix

In [ ]:
plt.figure(figsize=(11, 8))
sns.heatmap(paysim.select_dtypes('number').corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('PaySim — correlation matrix')
plt.show()

## Custom dataset — fraud rates by categorical features

In [ ]:
for col in ['Device_Used','Location','Payment_Method','Transaction_Type']:
    print(f'\n=== {col} ===')
    print(custom.groupby(col)['Fraudulent'].agg(['count','mean']).rename(columns={'mean':'fraud_rate'}).sort_values('fraud_rate', ascending=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.boxplot(data=custom, x='Fraudulent', y='Transaction_Amount', ax=axes[0])
axes[0].set_title('Amount — legit vs fraud')
axes[0].set_yscale('log')

sns.boxplot(data=custom, x='Fraudulent', y='Account_Age', ax=axes[1])
axes[1].set_title('Account age — legit vs fraud')
plt.tight_layout(); plt.show()

## Take-aways for modeling

- **PaySim fraud is concentrated in TRANSFER + CASH_OUT types.** Other types are essentially fraud-free.
- **`amount == oldbalanceOrg`** (origin account drained) is highly predictive of fraud.
- **`dest_was_empty == 1`** is correlated with fraud — money moving to fresh accounts.
- **Custom dataset:** `Previous_Fraudulent_Transactions`, low `Account_Age`, and `Payment_Method == Invalid Method` are the strongest signals.
- **Class imbalance** is severe on PaySim (~0.13% fraud); the next notebook handles it with SMOTE.